In [1]:
import numpy as np

In [ ]:
# import necessary libraries 
import numpy as np
from typing import Tuple, Optional


class Grid_Generation: 

    r"""
        
        Inputs
        ------
        smoothing_length : float
            Smoothing length (kernel size) for grid spacing calculations.
        particle_bounds : ndarray of shape (3, 2)
            Minimum and maximum coordinates for the particle domain along x, y, z.
        grid_dimensions : int
            Dimensionality of the grid: 1, 2, or 3.
        grid_axes : str
            Axes along which the grid will be generated. 
            For 1D: 'x', 'y', or 'z'.  
            For 2D: 'xy', 'xz', or 'yz'.  
            For 3D: 'xyz' (implicitly used).
        max_particle_diameter : float
            Maximum particle diameter used for domain buffer calculations.
        automatic_range : bool
            If True, automatically determine the grid range with domain padding.
        custom_grid_range : tuple of float or None
            Custom range (x0, x1, y0, y1, z0, z1) if `automatic_range` is False.
        custom_grid_transects : tuple of float or None
            Custom transect positions (x_transect, y_transect, z_transect) 
            if `automatic_range` is False.

        Example
        -------
        >>> grid_gen = Grid_Generation(smoothing_length=0.1, 
        ...                             particle_bounds=np.array([[0, 1], [0, 1], [0, 1]]), 
        ...                             grid_dimensions=2, 
        ...                             grid_axes='xy', 
        ...                             max_particle_diameter=0.05, 
        ...                             automatic_range=True, 
        ...                             custom_grid_range=(None, None, None, None, None, None), 
        ...                             custom_grid_transects=(None, None, None))
        >>> GridPoints, Nodes, Spacing, Ranges_Length = grid_gen.Generate()
        >>> print("Grid Points:\n", GridPoints)
        >>> print("Nodes:", Nodes)
        >>> print("Spacing:", Spacing)
        >>> print("Ranges Length:", Ranges_Length)

        """

    def __init__(self, smoothing_length:float, particle_bounds:np.ndarray, grid_dimensions:int, 
                 grid_axes:str, max_particle_diameter:float, automatic_range:bool, 
                 custom_grid_range:Tuple, custom_grid_transects:Tuple):
   
        
        # Validate smoothing_length
        if smoothing_length <= 0:
            raise ValueError(f"smoothing_length must be positive, got {smoothing_length}")
        
        # Validate particle_bounds
        if particle_bounds.shape != (3, 2):
            raise ValueError(f"particle_bounds must have shape (3, 2), got {particle_bounds.shape}")
        
        for i, axis in enumerate(['x', 'y', 'z']):
            if particle_bounds[i, 1] <= particle_bounds[i, 0]:
                raise ValueError(f"Invalid bounds for {axis}-axis: max ({particle_bounds[i, 1]}) must be > min ({particle_bounds[i, 0]})")
        
        # Validate grid_dimensions
        if grid_dimensions not in [1, 2, 3]:
            raise ValueError(f"grid_dimensions must be 1, 2, or 3, got {grid_dimensions}")
        
        # Validate grid_axes
        valid_axes = {1: ['x', 'y', 'z'], 2: ['xy', 'xz', 'yz'], 3: ['xyz']}
        if grid_axes not in valid_axes[grid_dimensions]:
            raise ValueError(f"For {grid_dimensions}D grid, axes must be one of {valid_axes[grid_dimensions]}, got '{grid_axes}'")
        
        # Validate max_particle_diameter
        if max_particle_diameter <= 0:
            raise ValueError(f"max_particle_diameter must be positive, got {max_particle_diameter}")
        
        # Validate custom ranges when automatic_range is False
        if not automatic_range:
            if custom_grid_range == (None,)*len(custom_grid_range):
                raise ValueError("custom_grid_range must be provided when automatic_range=False")
            if custom_grid_transects == (None,)*len(custom_grid_transects):
                raise ValueError("custom_grid_transects must be provided when automatic_range=False")
        
        self.c = smoothing_length
        self.bounds = particle_bounds
        self.dimensions = grid_dimensions
        self.axes = grid_axes
        self.dmax = max_particle_diameter
        self.automatic_range = automatic_range
        self.custom_grid_range = custom_grid_range
        self.custom_grid_transects = custom_grid_transects
    
    # Automatic range determination
    def Automatic_Range(self)->Tuple[list, list, list]:
        """
            Automatically determine the grid coordinate ranges 
            based on domain bounds and kernel size.

            Inputs
            -------
            x_range : list of float
                Minimum and maximum x-coordinates for the grid.
            y_range : list of float
                Minimum and maximum y-coordinates for the grid.
            z_range : list of float
                Minimum and maximum z-coordinates for the grid.

            Outputs
            -------
            x_range : list of float
                Minimum and maximum x-coordinates for the grid.
            y_range : list of float
                Minimum and maximum y-coordinates for the grid.
            z_range : list of float
                Minimum and maximum z-coordinates for the grid.

            Notes
            -----

            The method offsets the bounds by :math:`2.5\,c + 0.5\,d_\mathrm{max}`, where :math:`c` is the smoothing length and :math:`d_\mathrm{max}` is the maximum particle diameter, 
            to avoid boundary effects.
    
            
        """

        delta = 2.5 * self.c + 0.5 * self.dmax # distance from the boundary of the domain
        
        min = np.zeros(3) ; max = np.zeros(3)

        for i in range(3):
            min[i] = self.bounds[i,0] + delta
            max[i] = self.bounds[i,1] - delta
        
        x_range = [min[0], max[0]] 
        y_range = [min[1], max[1]] 
        z_range = [min[2], max[2]] 

        return x_range, y_range, z_range
    
    # Create grid points
    @staticmethod
    def Create_grid_points(X_range:list, Y_range:list, Z_range:list, 
                           X_transect:Optional[float], Y_transect:Optional[float], Z_transect:Optional[float], 
                           c:float, high_res_scaling:Optional[float]=1.5, 
                           dimensions:int=3, axes:str='xyz') -> Tuple[np.ndarray,np.ndarray,np.ndarray]:
        """
        Create structured grid points in 1D, 2D, or 3D.

        Inputs
        ------
        X_range, Y_range, Z_range : list of float or None
            Coordinate ranges for each axis, in the form [min, max]. 
            If None, that axis will be fixed at its transect value.
        X_transect, Y_transect, Z_transect : float or None
            Transect positions for fixed coordinates when not 
            generating points along that axis.
        c : float
            Smoothing length used for grid spacing calculation.
        high_res_scaling : float, optional
            Scaling factor for grid density (default is 1.5).
        dimensions : int, {1, 2, 3}
            Dimensionality of the generated grid.
        axes : str
            Axes along which to generate the grid. Options are:

                - **3D**: 'xyz'
                - **2D**: 'xy', 'xz', 'yz'
                - **1D**: 'x', 'y', 'z'

        Outputs
        -------
        grid_points : ndarray of shape (N, 3)
            Array of generated grid points in 3D coordinates.
        nodes : ndarray of shape (3,)
            Number of nodes along each axis (0 if fixed).
        spacing : ndarray
            Grid spacing along each active axis.

        Raises
        ------
        ValueError
            If the number of grid points in a direction is <= 1, 
            or if `dimensions`/`axes` combination is invalid.

        Notes
        -----

            - Spacing along active axes is computed as ``c / high_res_scaling``.

            - The output is always a set of points in 3D space, even for 1D and 2D grids, to maintain compatibility with 3D data structures.

            - This method uses `np.meshgrid` to create structured grids.

        """

        # General grid parameters
        # X - dimension
        if X_range is not None and all(v is not None for v in X_range):
            #print("X range",X_range)
            xmin, xmax = X_range # unpack the range
            Nx = int(np.ceil(high_res_scaling * (xmax - xmin) / c)+1) # calculate number of grid points
            if Nx <= 1:
                raise ValueError("The number of grid points in x direction is <= 1. Increase X_range")
            dx = (xmax - xmin) / (Nx - 1) # calculate spacing of the grid
            x = np.linspace(xmin,xmax,Nx) # generate grid points
        else: 
            x = None ; dx = None ; Nx = None
        # -------------------------------------------------
        # Y - dimension
        if Y_range is not None and all(v is not None for v in Y_range):
            ymin, ymax = Y_range # unpack the range
            #print("Y range",Y_range)
            Ny = int(np.ceil(high_res_scaling * (ymax - ymin) / c)+1) # calculate number of grid points
            if Ny <= 1:
                raise ValueError("The number of grid points in y direction is <= 1. Increase Y_range")
            dy = (ymax - ymin) / (Ny - 1) # calculate spacing of the grid
            y = np.linspace(ymin,ymax,Ny) # generate grid points
            #print("dy")
        else:
            y = None ; dy = None ; Ny = None
        # -------------------------------------------------
        # Z - dimension
        if Z_range is not None and all(v is not None for v in Z_range):
            #print("Z range" ,Z_range)
            zmin, zmax = Z_range # unpack the range
            #print(zmin, zmax)
            Nz = int(np.ceil(high_res_scaling * (zmax - zmin) / c)+1) # calculate number of grid points
            if Nz <= 1:
                raise ValueError("The number of grid points in z direction is <= 1. Increase Z_range")
            dz = (zmax - zmin) / (Nz - 1) # calculate spacing of the grid
            z = np.linspace(zmin,zmax,Nz) # generate grid points
        else:
            z = None ; dz = None ; Nz = None
        # -------------------------------------------------

        #                      * * * * 

        # ------------------------------------------------
        # 1-D grid
        if dimensions == 1:
            if axes == 'x':
                # Check that transect values are provided
                if x is None:
                    raise ValueError("Axes x: X_range is None, please provide a value.")
                if Y_transect is None:
                    raise ValueError("Axes x: Y_transect is None, please provide a value.")
                if Z_transect is None:
                    raise ValueError("Axes x: Z_transect is None, please provide a value.")
                # Generate x-coordinates
                grid_points_x = x 
                grid_points = np.column_stack((grid_points_x, np.full_like(grid_points_x, Y_transect), np.full_like(grid_points_x, Z_transect))) # Create 3D points using broadcasting
                nodes = np.array([Nx, 1, 1])
                spacing = np.array([dx])

            elif axes == 'y':
                # Check that transect values are provided
                if y is None:
                    raise ValueError("Axes y: Y_range is None, please provide a value.")
                if X_transect is None:
                    raise ValueError("Axes y: X_transect is None, please provide a value.")
                if Z_transect is None:
                    raise ValueError("Axes y: Z_transect is None, please provide a value.")
                # Generate y-coordinates
                grid_points_y = y
                grid_points = np.column_stack((np.full_like(grid_points_y, X_transect), grid_points_y, np.full_like(grid_points_y, Z_transect)))
                nodes = np.array([1, Ny, 1])
                spacing = np.array([dy])
            
            elif axes == 'z':
                # Check that transect values are provided
                if z is None:
                    raise ValueError("Axes z: Z_range is None, please provide a value.")
                if X_transect is None:
                    raise ValueError("Axes z: X_transect is None, please provide a value.")
                if Y_transect is None:
                    raise ValueError("Axes z: Y_transect is None, please provide a value.")
                # Generate z-coordinates
                grid_points_z = z
                grid_points = np.column_stack((np.full_like(grid_points_z, X_transect), np.full_like(grid_points_z, Y_transect), grid_points_z))
                nodes = np.array([1, 1, Nz])
                spacing = np.array([dz])
            
            else:
                raise ValueError("Invalid axes for 1D grid. Must be 'x', 'y', or 'z'.")
        # -------------------------------------------------
        # 2-D grid
        elif dimensions == 2:
            
            if axes == 'xy':
                # Check that transect values are provided
                if x is None:
                    raise ValueError("Axes xy: X_range is None, please provide a value.")
                if y is None:
                    raise ValueError("Axes xy: Y_range is None, please provide a value.")
                if Z_transect is None:
                    raise ValueError("Axes xy: Z_transect is None, please provide a value.")
                # Generate grid
                #print("xy 2d")
                xx, yy = np.meshgrid(x, y, indexing='ij')  # Generate grid
                x_cg = np.reshape(xx, (Nx * Ny))  # Reshape the grid points in x direction
                y_cg = np.reshape(yy, (Nx * Ny))  # Reshape the grid points in y direction
                z_cg = np.full_like(x_cg, Z_transect)  # Create a constant array for z
                grid_points = np.column_stack((x_cg, y_cg, z_cg))  # Combine x, y, and z
                nodes = np.array([Nx, Ny, 1])
                spacing = np.array([dx, dy]) 
                #print("done")
            
            elif axes == 'xz':
                # Check that transect values are provided
                if x is None:
                    raise ValueError("Axes xz: X_range is None, please provide a value.")
                if z is None:
                    raise ValueError("Axes xz: Z_range is None, please provide a value.")
                if Y_transect is None:
                    raise ValueError("Axes xz: Y_transect is None, please provide a value.")
                # Generate grid
                xx, zz = np.meshgrid(x, z, indexing='ij')  # Generate grid
                x_cg = np.reshape(xx, (Nx * Nz))  # Reshape the grid points in x direction
                z_cg = np.reshape(zz, (Nx * Nz))  # Reshape the grid points in z direction
                y_cg = np.full_like(x_cg, Y_transect)  # Create a constant array for y
                grid_points = np.column_stack((x_cg, y_cg, z_cg))  # Combine x, y, and z
                nodes = np.array([Nx, 1, Nz])
                spacing = np.array([dx, dz])
            
            elif axes == 'yz':
                # Check that transect values are provided
                if y is None:
                    raise ValueError("Axes yz: Y_range is None, please provide a value.")
                if z is None:  
                    raise ValueError("Axes yz: Z_range is None, please provide a value.")
                if X_transect is None:
                    raise ValueError("Axes yz: X_transect is None, please provide a value.")
                # Generate grid
                yy, zz = np.meshgrid(y, z, indexing='ij')  # Generate grid
                y_cg = np.reshape(yy, (Ny * Nz))  # Reshape the grid points in y direction
                z_cg = np.reshape(zz, (Ny * Nz))  # Reshape the grid points in z direction
                x_cg = np.full_like(y_cg, X_transect)  # Create a constant array for x
                grid_points = np.column_stack((x_cg, y_cg, z_cg))  # Combine x, y, and z
                nodes = np.array([1, Ny, Nz])
                spacing = np.array([dy, dz])
            
            else:
                raise ValueError("Invalid axes for 2D grid. Must be 'xy', 'xz', or 'yz'.")
        # --------------------------------------------------
        # 3-D grid
        elif dimensions == 3:
            xx,yy,zz = np.meshgrid(x,y,z,indexing='ij') # generate grid
            x_cg = np.reshape(xx,(Nx*Ny*Nz))
            y_cg = np.reshape(yy,(Nx*Ny*Nz))
            z_cg = np.reshape(zz,(Nx*Ny*Nz))
            grid_points = np.transpose([x_cg,y_cg,z_cg]) # prepare the array of CG points
            nodes = np.array([Nx, Ny, Nz])
            spacing = np.array([dx, dy, dz])
        # --------------------------------------------------
        # Raise error if dimensions are not 1, 2, or 3
        else:
            raise ValueError("Invalid dimensions. Must be 1, 2, or 3.")
                
        return grid_points, nodes, spacing
    
    # Make grid
    def Generate(self)->Tuple[np.ndarray,np.ndarray,np.ndarray,np.ndarray]:
        """
        Generate the computational grid points.

        Outputs
        -------
        GridPoints : ndarray of shape (N, 3)
            Generated grid points in 3D coordinates.
        Nodes : ndarray of shape (3,)
            Number of nodes along each axis.
        Spacing : ndarray
            Grid spacing along each active axis.
        Ranges_Length : ndarray of shape (3,)
            Length of each coordinate range. Zero if the axis is fixed.

        Notes
        -----
            - If `automatic_range` is True, grid ranges are computed using :meth:`Automatic_Range` with domain padding.

            - If `automatic_range` is False, ranges and transects are taken from `custom_grid_range` and `custom_grid_transects`.

            - The method calls :meth:`Create_grid_points` to build the grid.

        """
     
        # 1. get ranges and or transects
        if self.automatic_range == True:
            print("Generating Grid with Automatic Grid Ranges")
            x_range_, y_range_, z_range_ = self.Automatic_Range()
            # 1D grid
            if self.dimensions == 1: 
                if self.axes == 'x': 
                    y_transect = y_range_[0] + 0.5 * (y_range_[1] - y_range_[0]) ; y_range = None
                    z_transect = z_range_[0] + 0.5 * (z_range_[1] - z_range_[0]) ; z_range = None
                    x_transect = None ; x_range = x_range_
                    print(f"Automatic bounds: x_range = {x_range}, y_transect = {y_transect}, z_transect = {z_transect}")
                elif self.axes == 'y':
                    x_transect = x_range_[0] + 0.5 * (x_range_[1] - x_range_[0]) ; x_range = None
                    z_transect = z_range_[0] + 0.5 * (z_range_[1] - z_range_[0]) ; z_range = None
                    y_transect = None ; y_range = y_range_
                    print(f"Automatic bounds: y_range = {y_range}, x_transect = {x_transect}, z_transect = {z_transect}")
                elif self.axes == 'z':
                    x_transect = x_range_[0] + 0.5 * (x_range_[1] - x_range_[0]) ; x_range = None
                    y_transect = y_range_[0] + 0.5 * (y_range_[1] - y_range_[0]) ; y_range = None
                    z_transect = None ; z_range = z_range_
                    print(f"Automatic bounds: z_range = {z_range}, x_transect = {x_transect}, y_transect = {y_transect}")
                else:
                    raise ValueError("Invalid axes for 1D grid. Must be 'x', 'y', or 'z'.")
            # 2D grid
            elif self.dimensions == 2:
                if self.axes == 'xy':
                    z_transect = z_range_[0] + 0.5 * (z_range_[1] - z_range_[0]) ; z_range = None
                    x_transect = None ; x_range = x_range_
                    y_transect = None ; y_range = y_range_
                    print(f"Automatic bounds: x_range = {x_range}, y_range = {y_range}, z_transect = {z_transect}")
                elif self.axes == 'xz':
                    y_transect = y_range_[0] + 0.5 * (y_range_[1] - y_range_[0]) ; y_range = None
                    z_transect = None ; z_range = z_range_
                    x_transect = None ; x_range = x_range_
                    print(f"Automatic bounds: x_range = {x_range}, y_transect = {y_transect}, z_range = {z_range}")
                elif self.axes == 'yz':
                    x_transect = x_range_[0] + 0.5 * (x_range_[1] - x_range_[0]) ; x_range = None
                    z_transect = None ; z_range = z_range_
                    y_transect = None ; y_range = y_range_
                    print(f"Automatic bounds: y_range = {y_range}, x_transect = {x_transect}, z_range = {z_range}")
                else:
                    raise ValueError("Invalid axes for 2D grid. Must be 'xy', 'xz', or 'yz'.")
            elif self.dimensions == 3:
                x_range = x_range_ ; y_range = y_range_ ; z_range = z_range_
                x_transect = None ; y_transect = None ; z_transect = None
                print(f"Automatic bounds: x_range = {x_range}, y_range = {y_range}, z_range = {z_range}")
                        
        elif self.automatic_range == False:
            print("Generating Grid with Customised Grid Ranges")
            x_0, x_1, y_0, y_1, z_0, z_1 = self.custom_grid_range
            x_tr, y_tr, z_tr = self.custom_grid_transects
            # flag that the user has not provided ranges bigger than the domain
            # b = self.bounds
            # print(f" bounds {b}")
            # # Adjust ranges and checks based on dimensions
            # if x_0 and x_1 is not None:
            #     if x_0 < b[0,0] or x_1 > b[0,1]:
            #         raise ValueError("The provided X ranges are bigger than the domain.")
            # if y_0 and y_1 is not None:
            #     if y_0 < b[1,0] or y_1 > b[1,1]:
            #         raise ValueError("The provided Y ranges are bigger than the domain.")
            # if z_0 and z_1 is not None:
            #     if z_0 < b[2,0] or z_1 > b[2,1]:
            #         raise ValueError("The provided Z ranges are bigger than the domain.")
            # # Check that transect values are provided
            # if x_tr is not None: 
            #     if x_tr < b[0,0] or x_tr > b[0,1]:
            #         raise ValueError("The provided X transect is outside the domain.")
            # if y_tr is not None:
            #     if y_tr < b[1,0] or y_tr > b[1,1]:
            #         raise ValueError("The provided Y transect is outside the domain.")
            # if z_tr is not None:
            #     if z_tr < b[2,0] or z_tr > b[2,1]:
            #         raise ValueError("The provided Z transect is outside the domain.")
            # put together the ranges and/or transects
            
            x_range = [x_0, x_1] ; y_range = [y_0, y_1] ; z_range = [z_0, z_1]
            y_transect = y_tr ; x_transect = x_tr ; z_transect = z_tr
            print(f"Customised grid bounds: x = {x_range}, y = {y_range}, z = {z_range}, x_transect = {x_transect}, y_transect = {y_transect}, z_transect = {z_transect}")

        # 2. generate the CG grid points
        GridPoints , Nodes, Spacing = self.Create_grid_points(X_range=x_range, Y_range=y_range, Z_range=z_range, 
                            X_transect = x_transect, Y_transect = y_transect, Z_transect = z_transect,
                            c = self.c, 
                            high_res_scaling = 2, 
                            dimensions = self.dimensions, axes = self.axes) 
    
        # 3. calculate the ranges length
        # Ranges_Length = np.array([
        #                 0 if np.all(r) is None else r[1] - r[0] 
        #                 for r in [x_range, y_range, z_range]
        #             ])
        Ranges_Length = np.array([
        0 if r is None or any(v is None for v in r) else r[1] - r[0]
        for r in [x_range, y_range, z_range]
    ])
                
        return GridPoints, Nodes, Spacing, Ranges_Length


<>:125: SyntaxWarning: invalid escape sequence '\,'
<>:125: SyntaxWarning: invalid escape sequence '\,'
/tmp/ipykernel_1088079/4025513029.py:125: SyntaxWarning: invalid escape sequence '\,'
  The method offsets the bounds by :math:`2.5\,c + 0.5\,d_\mathrm{max}`, where :math:`c` is the smoothing length and :math:`d_\mathrm{max}` is the maximum particle diameter,


In [22]:
# Example 1: 3D Grid with Automatic Range
# =======================================
print("=== Example 1: 3D Grid with Automatic Range ===")

# Define particle domain bounds (min, max for each axis)
particle_bounds = np.array([
    [0.0, 10.0],  # x-bounds
    [0.0, 5.0],   # y-bounds
    [0.0, 8.0]    # z-bounds
])

# Create 3D grid generator
grid_gen_3d = Grid_Generation(
    smoothing_length=0.5,
    particle_bounds=particle_bounds,
    grid_dimensions=3,
    grid_axes='xy',
    max_particle_diameter=0.2,
    automatic_range=True,
    custom_grid_range=(None, None, None, None, None, None),
    custom_grid_transects=(None, None, None)
)

# Generate the grid
GridPoints_3d, Nodes_3d, Spacing_3d, Ranges_Length_3d = grid_gen_3d.Generate()

print(f"3D Grid:")
print(f"  Grid Points Shape: {GridPoints_3d.shape}")
print(f"  Nodes: {Nodes_3d}")
print(f"  Spacing: {Spacing_3d}")
print(f"  Range Lengths: {Ranges_Length_3d}")
print()

=== Example 1: 3D Grid with Automatic Range ===
hereeee


/tmp/ipykernel_1088079/4025513029.py:125: SyntaxWarning: invalid escape sequence '\,'
  The method offsets the bounds by :math:`2.5\,c + 0.5\,d_\mathrm{max}`, where :math:`c` is the smoothing length and :math:`d_\mathrm{max}` is the maximum particle diameter,


ValueError: For 3D grid, axes must be one of ['xyz'], got 'xy'

In [9]:
# Example 2: 2D Grid (XY plane) with Custom Range
# ===============================================
print("=== Example 2: 2D Grid (XY plane) with Custom Range ===")

grid_gen_2d = Grid_Generation(
    smoothing_length=0.3,
    particle_bounds=particle_bounds,  # Same domain
    grid_dimensions=2,
    grid_axes='xy',
    max_particle_diameter=0.15,
    automatic_range=False,
    custom_grid_range=(1.0, 9.0, 0.5, 4.5, None, None),  # x0, x1, y0, y1, z0, z1
    custom_grid_transects=(None, None, 4.0)  # z-transect at z=4.0
)

GridPoints_2d, Nodes_2d, Spacing_2d, Ranges_Length_2d = grid_gen_2d.Generate()

print(f"2D Grid (XY plane at z=4.0):")
print(f"  Grid Points Shape: {GridPoints_2d.shape}")
print(f"  Nodes: {Nodes_2d}")
print(f"  Spacing: {Spacing_2d}")
print(f"  Range Lengths: {Ranges_Length_2d}")
print()

=== Example 2: 2D Grid (XY plane) with Custom Range ===
Generating Grid with Customised Grid Ranges
Customised grid bounds: x = [1.0, 9.0], y = [0.5, 4.5], z = [None, None], x_transect = None, y_transect = None, z_transect = 4.0
2D Grid (XY plane at z=4.0):
  Grid Points Shape: (1540, 3)
  Nodes: [55 28  1]
  Spacing: [0.14814815 0.14814815]
  Range Lengths: [8. 4. 0.]



In [10]:
# Example 3: 1D Grid (X-axis) with Automatic Range
# ================================================
print("=== Example 3: 1D Grid (X-axis) with Automatic Range ===")

grid_gen_1d = Grid_Generation(
    smoothing_length=0.4,
    particle_bounds=particle_bounds,
    grid_dimensions=1,
    grid_axes='x',
    max_particle_diameter=0.1,
    automatic_range=True,
    custom_grid_range=(None, None, None, None, None, None),
    custom_grid_transects=(None, 2.5, 4.0)  # y-transect=2.5, z-transect=4.0
)

GridPoints_1d, Nodes_1d, Spacing_1d, Ranges_Length_1d = grid_gen_1d.Generate()

print(f"1D Grid (X-axis at y=2.5, z=4.0):")
print(f"  Grid Points Shape: {GridPoints_1d.shape}")
print(f"  Nodes: {Nodes_1d}")
print(f"  Spacing: {Spacing_1d}")
print(f"  Range Lengths: {Ranges_Length_1d}")
print()

=== Example 3: 1D Grid (X-axis) with Automatic Range ===
Generating Grid with Automatic Grid Ranges
Automatic bounds: x_range = [np.float64(1.05), np.float64(8.95)], y_transect = 2.5, z_transect = 4.0
1D Grid (X-axis at y=2.5, z=4.0):
  Grid Points Shape: (41, 3)
  Nodes: [41  1  1]
  Spacing: [0.1975]
  Range Lengths: [7.9 0.  0. ]



In [11]:
# Example 4: Small Domain Example (for testing edge cases)
# ========================================================
print("=== Example 4: Small Domain Example ===")

small_bounds = np.array([
    [0.0, 2.0],   # x-bounds
    [0.0, 1.5],   # y-bounds
    [0.0, 1.0]    # z-bounds
])

grid_gen_small = Grid_Generation(
    smoothing_length=0.1,  # Small smoothing length
    particle_bounds=small_bounds,
    grid_dimensions=2,
    grid_axes='xz',
    max_particle_diameter=0.05,
    automatic_range=True,
    custom_grid_range=(None, None, None, None, None, None),
    custom_grid_transects=(None, 0.75, None)  # y-transect at middle
)

GridPoints_small, Nodes_small, Spacing_small, Ranges_Length_small = grid_gen_small.Generate()

print(f"Small Domain 2D Grid (XZ plane at y=0.75):")
print(f"  Grid Points Shape: {GridPoints_small.shape}")
print(f"  Nodes: {Nodes_small}")
print(f"  Spacing: {Spacing_small}")
print(f"  Range Lengths: {Ranges_Length_small}")

=== Example 4: Small Domain Example ===
Generating Grid with Automatic Grid Ranges
Automatic bounds: x_range = [np.float64(0.275), np.float64(1.725)], y_transect = 0.75, z_range = [np.float64(0.275), np.float64(0.725)]
Small Domain 2D Grid (XZ plane at y=0.75):
  Grid Points Shape: (310, 3)
  Nodes: [31  1 10]
  Spacing: [0.04833333 0.05      ]
  Range Lengths: [1.45 0.   0.45]
